# Análise de Ataque — dos_noti_flood.pcap

Entrada: arquivo `.pcap` — sem exports manuais do Wireshark.  
Ferramenta: `tshark` (CLI do Wireshark) chamado via `subprocess`.  
Todos os gráficos e números são derivados do PCAP em tempo de execução.

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'plotly', '-q'])

# ── Configuração ────────────────────────────────────────────────────────────
TSHARK = r'C:\Program Files\Wireshark\tshark.exe'
PCAP   = r'C:\Mestrado\SDV_Research\experiments\notebooks\data\pcap\dos_noti_flood.pcap'

import io, collections
import pandas as pd
import numpy  as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML
import warnings; warnings.filterwarnings('ignore')

BG    = '#0f1117'; PANEL = '#161b2e'; GRID = '#2a3550'
TEXT  = '#d0d8f0'; A = '#4e7fff';  G = '#7bff9c'
T     = '#7bffd9'; Y = '#ffd97b';  P = '#c07bff'
O     = '#ff9d7b'; R = '#ff7b7b'

LB = dict(paper_bgcolor=BG, plot_bgcolor=PANEL,
          font=dict(color=TEXT, family='monospace'),
          title_font=dict(color=TEXT, size=15))

def ha(h, a=1.0):
    r,g,b = int(h[1:3],16), int(h[3:5],16), int(h[5:7],16)
    return f'rgba({r},{g},{b},{a})'

_first = True
def show(fig):
    global _first
    display(HTML(fig.to_html(full_html=False,
                             include_plotlyjs='cdn' if _first else False)))
    _first = False

print('OK')

OK


In [2]:
# ── Extração via tshark ─────────────────────────────────────────────────────
# --enable-protocol someip       — garante que o dissector SOME/IP está ativo
# --enable-heuristic someip_tcp_heur — detecta SOME/IP em portas TCP não-padrão
# --enable-heuristic someip_udp_heur — detecta SOME/IP em portas UDP não-padrão

FIELDS = [
    'frame.number',
    'frame.time_relative',
    'frame.len',
    '_ws.col.Protocol',
    'ip.src',
    'ip.dst',
    'ip.proto',
    'ip.ttl',
    'tcp.srcport',
    'tcp.dstport',
    'udp.srcport',
    'udp.dstport',
    'someip.serviceid',
    'someip.methodid',
    'someip.messagetype',
    'someip.returncode',
    'someip.length',
    'someipsd.entry.type',
]

cmd = [
    TSHARK,
    '--enable-protocol',  'someip',
    '--enable-heuristic', 'someip_tcp_heur',
    '--enable-heuristic', 'someip_udp_heur',
    '-r', PCAP,
    '-T', 'fields',
    '-E', 'header=y', '-E', 'separator=,', '-E', 'occurrence=f',
]
for f in FIELDS:
    cmd += ['-e', f]

print('Executando tshark...')
result = subprocess.run(cmd, capture_output=True, text=True, encoding='utf-8')
if result.returncode != 0:
    print('ERRO:', result.stderr[:500])
else:
    df_raw = pd.read_csv(io.StringIO(result.stdout), low_memory=False)
    print(f'Extraídos: {len(df_raw):,} pacotes  |  colunas: {list(df_raw.columns)}')

Executando tshark...


Extraídos: 1,931,593 pacotes  |  colunas: ['frame.number', 'frame.time_relative', 'frame.len', '_ws.col.protocol', 'ip.src', 'ip.dst', 'ip.proto', 'ip.ttl', 'tcp.srcport', 'tcp.dstport', 'udp.srcport', 'udp.dstport', 'someip.serviceid', 'someip.methodid', 'someip.messagetype', 'someip.returncode', 'someip.length', 'someipsd.entry.type']


In [3]:
# ── Limpeza e tipagem ────────────────────────────────────────────────────────
df = df_raw.copy()
df.columns = [
    'num', 'time', 'frame_len', 'proto',
    'src_ip', 'dst_ip', 'ip_proto', 'ttl',
    'tcp_sport', 'tcp_dport',
    'udp_sport', 'udp_dport',
    'svc_id', 'meth_id', 'msg_type', 'ret_code', 'someip_len',
    'sd_entry_type',
]

df['time']      = pd.to_numeric(df['time'],      errors='coerce')
df['frame_len'] = pd.to_numeric(df['frame_len'], errors='coerce')
df['ttl']       = pd.to_numeric(df['ttl'],       errors='coerce')
df['tcp_dport'] = pd.to_numeric(df['tcp_dport'], errors='coerce')
df['tcp_sport'] = pd.to_numeric(df['tcp_sport'], errors='coerce')
df['udp_dport'] = pd.to_numeric(df['udp_dport'], errors='coerce')
df['udp_sport'] = pd.to_numeric(df['udp_sport'], errors='coerce')

for c in ['svc_id', 'meth_id', 'msg_type', 'ret_code']:
    df[c] = df[c].astype(str).str.strip().str.lower()
    df[c] = df[c].where(df[c] != 'nan', other='')

df_ip     = df[df['src_ip'].notna() & (df['src_ip'] != '')]
df_someip = df[df['svc_id'] != ''].copy()
df_sd     = df[df['proto'].str.contains('SOMEIPSD|SOME/IP-SD', case=False, na=False)]

print(f'Total frames    : {len(df):>10,}')
print(f'Frames IP       : {len(df_ip):>10,}')
print(f'Frames SOME/IP  : {len(df_someip):>10,}')
print(f'Frames SOME/IP-SD: {len(df_sd):>9,}')
print()
print('Protocolos únicos encontrados:')
print(df['proto'].value_counts().to_string())

Total frames    :  1,931,593
Frames IP       :  1,930,421
Frames SOME/IP  :    973,422
Frames SOME/IP-SD:   305,914

Protocolos únicos encontrados:
proto
TCP           891108
SOME/IP       667508
SOME/IP-SD    305914
IGMPv3         65891
ARP             1172


## 1. Hierarquia de Protocolos

In [4]:
# ── Reconstruir hierarquia igual ao Wireshark ────────────────────────────────
# Wireshark conta cada pacote em TODOS os níveis que ele atravessa (árvore).
# SOME/IP-SD é FILHO de SOME/IP (UDP) — o pai conta SD + não-SD.

ip_proto_num = pd.to_numeric(df['ip_proto'], errors='coerce')
someip_mask  = df['proto'].str.contains('SOME/IP', na=False, case=False) & \
               ~df['proto'].str.contains('SD',     na=False, case=False)

# Contagens por nível
n_frame   = len(df)
n_eth     = len(df)
n_ipv4    = int(df['src_ip'].notna().sum())
n_udp     = int(df['udp_dport'].notna().sum())
n_tcp     = int(df['tcp_dport'].notna().sum())
n_igmp    = int((df['proto'] == 'IGMPv3').sum())
n_arp     = int((df['proto'] == 'ARP').sum())

n_someip_tcp     = int((someip_mask & (ip_proto_num == 6)).sum())
n_someipsd       = int(df['proto'].str.contains('SOME/IP-SD', na=False, case=False).sum())
n_someip_udp_end = int((someip_mask & (ip_proto_num == 17)).sum())  # não-SD UDP SOME/IP
n_someip_udp     = n_someip_udp_end + n_someipsd                    # total UDP SOME/IP (pai do SD)

n_tcp_ctrl = n_tcp - n_someip_tcp

# ── Tabela no estilo Wireshark ────────────────────────────────────────────────
rows = [
    # (indent, protocolo,                         total,       end_packets)
    (0, 'Frame',                                  n_frame,     0),
    (1, 'Ethernet',                               n_eth,       0),
    (2, 'Internet Protocol Version 4',            n_ipv4,      0),
    (3, 'User Datagram Protocol',                 n_udp,       0),
    (4, 'SOME/IP Protocol',                       n_someip_udp,  n_someip_udp_end),
    (5, 'SOME/IP Service Discovery Protocol',     n_someipsd,  n_someipsd),
    (3, 'Transmission Control Protocol',          n_tcp,       n_tcp_ctrl),
    (4, 'SOME/IP Protocol',                       n_someip_tcp, n_someip_tcp),
    (3, 'Internet Group Management Protocol',     n_igmp,      n_igmp),
    (2, 'Address Resolution Protocol',            n_arp,       n_arp),
]

print(f'{"Protocol":<54} {"Packets":>10}  {"End Packets":>12}')
print('─' * 80)
for indent, proto, pkts, end in rows:
    prefix  = '  ' * indent + ('└── ' if indent > 0 else '')
    pct     = pkts / n_frame * 100
    end_str = f'{end:,}' if end > 0 else '—'
    print(f'{prefix}{proto:<{54 - len(prefix)}} {pkts:>10,} ({pct:5.1f}%)  {end_str:>12}')

# Verificação
total_leaf = n_someip_tcp + n_tcp_ctrl + n_someipsd + n_someip_udp_end + n_igmp + n_arp
print(f'\nSoma end packets: {total_leaf:,}  (esperado: {n_frame:,})  {"OK" if total_leaf == n_frame else "DIVERGE"}')

# ── Gráfico de pizza (leaf protocols) ────────────────────────────────────────
leaf_data = [
    ('SOME/IP (TCP)',  n_someip_tcp),
    ('TCP ctrl/ACK',  n_tcp_ctrl),
    ('SOME/IP-SD',    n_someipsd),
    ('IGMPv3',        n_igmp),
    ('SOME/IP (UDP)', n_someip_udp_end),
    ('ARP',           n_arp),
]
labels  = [x[0] for x in leaf_data]
values  = [x[1] for x in leaf_data]
palette = [R, A, Y, G, T, P]

fig = go.Figure(go.Pie(
    labels=labels, values=values,
    hole=0.4, marker_colors=palette,
    textinfo='label+percent',
    hovertemplate='%{label}<br>%{value:,} pkts<extra></extra>'
))
fig.update_layout(**LB,
    title=f'Protocolo folha — {n_frame:,} frames totais',
    height=440)
show(fig)

Protocol                                                  Packets   End Packets
────────────────────────────────────────────────────────────────────────────────
Frame                                                   1,931,593 (100.0%)             —
  └── Ethernet                                          1,931,593 (100.0%)             —
    └── Internet Protocol Version 4                     1,930,421 ( 99.9%)             —
      └── User Datagram Protocol                          308,508 ( 16.0%)             —
        └── SOME/IP Protocol                              308,508 ( 16.0%)         2,594
          └── SOME/IP Service Discovery Protocol          305,914 ( 15.8%)       305,914
      └── Transmission Control Protocol                 1,556,022 ( 80.6%)       891,108
        └── SOME/IP Protocol                              664,914 ( 34.4%)       664,914
      └── Internet Group Management Protocol               65,891 (  3.4%)        65,891
    └── Address Resolution Protocol   

## 2. IPs da Rede — Volume de Tráfego

In [5]:
sent = df_ip.groupby('src_ip').size().rename('Enviados')
recv = df_ip.groupby('dst_ip').size().rename('Recebidos')
ips  = pd.concat([sent, recv], axis=1).fillna(0).astype(int)
ips  = ips.sort_values('Enviados', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(name='Enviados',  x=ips.index, y=ips['Enviados'],
    marker_color=R, hovertemplate='%{x}<br>Enviados: %{y:,}<extra></extra>'))
fig.add_trace(go.Bar(name='Recebidos', x=ips.index, y=ips['Recebidos'],
    marker_color=ha(R, 0.5), hovertemplate='%{x}<br>Recebidos: %{y:,}<extra></extra>'))
fig.update_layout(**LB, title='Pacotes enviados/recebidos por IP',
    barmode='group', height=420,
    xaxis=dict(title='IP', gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID),
    legend=dict(bgcolor=PANEL, bordercolor=GRID))
show(fig)
print(ips.to_string())

                 Enviados  Recebidos
172.18.0.5         421901     420503
172.18.0.11        372817     350782
172.18.0.2         333938     331499
172.18.0.9         189803     151459
172.18.0.4         181316     180311
172.18.0.6         159392     159576
172.18.0.8         130766     131076
172.18.0.7          69448      31193
172.18.0.3          69285      31125
172.18.0.10          1755       1006
224.0.0.22              0      65891
224.244.224.245         0      76000


## 3. Distribuição de Tamanho de Frames

In [6]:
sizes = df['frame_len'].dropna()

fig = go.Figure(go.Histogram(
    x=sizes, nbinsx=60,
    marker_color=R, marker_line_color=GRID, marker_line_width=0.5,
    hovertemplate='%{x} bytes<br>%{y:,} frames<extra></extra>'
))
fig.update_layout(**LB, title='Distribuição de tamanho de frame (Ethernet)',
    xaxis=dict(title='Tamanho (bytes)', gridcolor=GRID),
    yaxis=dict(title='Frequência', gridcolor=GRID), height=380)
show(fig)

print(f'Min    : {int(sizes.min())} bytes')
print(f'Max    : {int(sizes.max())} bytes')
print(f'Média  : {sizes.mean():.1f} bytes')
print(f'Mediana: {sizes.median():.0f} bytes')
print()
bins   = [0, 79, 159, 319, 9999]
labels = ['<=79', '80-159', '160-319', '320+']
faixas = pd.cut(sizes, bins=bins, labels=labels)
print('Contagem por faixa:')
print(faixas.value_counts().sort_index().to_string())

Min    : 42 bytes
Max    : 122 bytes
Média  : 84.8 bytes
Mediana: 86 bytes

Contagem por faixa:
frame_len
<=79       960088
80-159     971505
160-319         0
320+            0


## 4. SOME/IP — Flows por Service ID, Method ID e Msg Type

In [7]:
MSG_TYPE = {
    '0x00': 'REQUEST',        '0x01': 'REQUEST_NO_RETURN',
    '0x02': 'NOTIFICATION',   '0x40': 'REQUEST_ACK',
    '0x41': 'RQST_NRET_ACK',  '0x42': 'NOTIF_ACK',
    '0x80': 'RESPONSE',       '0x81': 'ERROR',
    '0xc0': 'RESPONSE_ACK',   '0xc1': 'ERROR_ACK',
}

df_someip['msg_type_name'] = df_someip['msg_type'].map(MSG_TYPE).fillna(df_someip['msg_type'])

flows = (df_someip
    .groupby(['svc_id', 'meth_id', 'src_ip', 'msg_type_name'])
    .size()
    .reset_index(name='pkts')
    .sort_values('pkts', ascending=False)
)

top_flows = flows.head(15)
labels = [f"{r['svc_id']}\n{r['meth_id']}\n{r['src_ip']}" for _, r in top_flows.iterrows()]

# Colorir por msg_type: NOTIFICATION em vermelho (padrão de flood)
type_color = {'NOTIFICATION': R, 'REQUEST': A, 'RESPONSE': G,
              'REQUEST_NO_RETURN': Y, 'ERROR': O}
bar_colors = [type_color.get(r['msg_type_name'], P) for _, r in top_flows.iterrows()]

fig = go.Figure(go.Bar(
    x=labels, y=top_flows['pkts'],
    marker_color=bar_colors,
    text=[r['msg_type_name'] for _, r in top_flows.iterrows()],
    textposition='outside',
    hovertemplate='%{x}<br>%{y:,} pkts<extra></extra>',
))
fig.update_layout(**LB,
    title='Top 15 flows SOME/IP por (Service ID, Method ID, IP Fonte)',
    xaxis=dict(title='Service / Method / IP', tickangle=30, gridcolor=GRID),
    yaxis=dict(title='Pacotes (log)', gridcolor=GRID, type='log'),
    height=500, showlegend=False)
show(fig)

print(flows.to_string(index=False))

svc_id meth_id      src_ip msg_type_name   pkts
0xffff  0x8100 172.18.0.11  NOTIFICATION 182467
0x1002  0x0003  172.18.0.5  NOTIFICATION 130328
0x1002  0x0001  172.18.0.5  NOTIFICATION 129296
0x1002  0x0002  172.18.0.5  NOTIFICATION  98744
0xffff  0x8100  172.18.0.2  NOTIFICATION  71925
0x1001  0x0001 172.18.0.11  NOTIFICATION  69393
0x1002  0x0004  172.18.0.5  NOTIFICATION  59376
0x1003  0x0003  172.18.0.4  NOTIFICATION  59334
0x1003  0x0002  172.18.0.4  NOTIFICATION  59334
0x1003  0x0001  172.18.0.4  NOTIFICATION  59109
0xffff  0x8100  172.18.0.9  NOTIFICATION  14702
0xffff  0x8100  172.18.0.7  NOTIFICATION  13082
0xffff  0x8100  172.18.0.3  NOTIFICATION  13013
0xffff  0x8100  172.18.0.5  NOTIFICATION   3732
0xffff  0x8100  172.18.0.4  NOTIFICATION   2989
0xffff  0x8100 172.18.0.10  NOTIFICATION   1753
0xffff  0x8100  172.18.0.6  NOTIFICATION   1500
0xffff  0x8100  172.18.0.8  NOTIFICATION    751
0x1002  0x0001  172.18.0.2       REQUEST    559
0x1003  0x0001  172.18.0.2       REQUEST

In [8]:
# Message Type — distribuição geral
mt_counts = df_someip['msg_type_name'].value_counts().reset_index()
mt_counts.columns = ['Msg Type', 'Pacotes']

mt_colors = [type_color.get(t, P) for t in mt_counts['Msg Type']]

fig = go.Figure(go.Bar(
    x=mt_counts['Msg Type'], y=mt_counts['Pacotes'],
    marker_color=mt_colors,
    text=[f'{v:,}' for v in mt_counts['Pacotes']],
    textposition='outside',
    hovertemplate='%{x}<br>%{y:,}<extra></extra>'
))
fig.update_layout(**LB, title='Distribuição de Message Type — SOME/IP',
    xaxis=dict(title='Message Type', gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID),
    height=400, showlegend=False)
show(fig)

total_si = len(df_someip)
for _, r in mt_counts.iterrows():
    print(f"  {r['Msg Type']:<22} {r['Pacotes']:>10,}  ({r['Pacotes']/total_si*100:.1f}%)")

  NOTIFICATION              970,828  (99.7%)
  REQUEST                     1,632  (0.2%)
  RESPONSE                      962  (0.1%)


## 5. Frame Size por (Service ID, Method ID)

In [9]:
fs = (df_someip
    .groupby(['svc_id', 'meth_id'])['frame_len']
    .agg(['min', 'max', 'mean', 'std', 'count'])
    .reset_index()
)
fs['tipo'] = fs.apply(lambda r: 'fixo' if r['min'] == r['max'] else 'variavel', axis=1)
fs = fs.sort_values(['svc_id', 'meth_id'])

fig = go.Figure(go.Table(
    header=dict(
        values=['Service ID', 'Method ID', 'Min (B)', 'Max (B)', 'Media (B)', 'Pacotes', 'Tipo'],
        fill_color=PANEL, font=dict(color=A, size=12), align='left'),
    cells=dict(
        values=[
            fs['svc_id'], fs['meth_id'],
            fs['min'].astype(int), fs['max'].astype(int),
            fs['mean'].round(1), fs['count'].apply(lambda x: f'{x:,}'),
            fs['tipo'],
        ],
        fill_color=[[BG if i%2==0 else PANEL for i in range(len(fs))]]*7,
        font=dict(color=[[G if t=='fixo' else O for t in fs['tipo']] if c==6 else [TEXT]*len(fs)
                         for c in range(7)], size=11),
        align='left')
))
fig.update_layout(**LB, title='Frame size por (Service ID, Method ID)', height=420)
show(fig)

## 6. SOME/IP-SD — Service Discovery

In [10]:
SD_TYPE = {'0': 'Find', '1': 'Offer', '6': 'Subscribe', '7': 'SubscribeAck'}

sd_rows = []
for _, row in df_sd.iterrows():
    types_raw = str(row.get('sd_entry_type', '') or '')
    for t in types_raw.split(','):
        t = t.strip()
        if t:
            sd_rows.append({'ip': row['src_ip'], 'type_raw': t,
                            'type_name': SD_TYPE.get(t, f'type_{t}')})

if sd_rows:
    df_sd_flat = pd.DataFrame(sd_rows)
    sd_pivot = (df_sd_flat
        .groupby(['ip', 'type_name'])
        .size()
        .reset_index(name='count')
        .pivot_table(index='ip', columns='type_name', values='count', fill_value=0)
    )
    type_colors = {'Find': Y, 'Offer': G, 'Subscribe': A, 'SubscribeAck': T}
    fig = go.Figure()
    for t in ['Find', 'Offer', 'Subscribe', 'SubscribeAck']:
        if t in sd_pivot.columns:
            fig.add_trace(go.Bar(name=t, x=sd_pivot.index, y=sd_pivot[t],
                marker_color=type_colors.get(t, P),
                hovertemplate=f'{t}: %{{y:,}}<extra></extra>'))
    fig.update_layout(**LB, title='SOME/IP-SD — entradas por tipo e IP',
        barmode='stack', height=420,
        xaxis=dict(title='IP', gridcolor=GRID),
        yaxis=dict(title='Entradas SD', gridcolor=GRID),
        legend=dict(bgcolor=PANEL, bordercolor=GRID))
    show(fig)
    print(sd_pivot.to_string())
else:
    print('Nenhuma entrada SOME/IP-SD encontrada neste PCAP.')
    print(f'Total frames SD: {len(df_sd):,}')

type_name    type_0x00  type_0x01  type_0x06  type_0x07
ip                                                     
172.18.0.10        0.0      747.0        0.0     1006.0
172.18.0.11        0.0    73743.0        0.0   108724.0
172.18.0.2         1.0        0.0    71924.0        0.0
172.18.0.3         1.0        0.0    13012.0        0.0
172.18.0.4         0.0      748.0        0.0     2241.0
172.18.0.5         0.0      748.0        0.0     2984.0
172.18.0.6         4.0        0.0     1496.0        0.0
172.18.0.7         4.0        0.0    13078.0        0.0
172.18.0.8         4.0        0.0      747.0        0.0
172.18.0.9         4.0        0.0    14698.0        0.0


## 7. Portas TCP/UDP — Destino

In [11]:
df_tcp = df[df['tcp_dport'].notna()].copy()
df_tcp['tcp_dport'] = df_tcp['tcp_dport'].astype(int)

top_ports = (df_tcp
    .groupby(['dst_ip', 'tcp_dport'])
    .size()
    .reset_index(name='pkts')
    .sort_values('pkts', ascending=False)
    .head(15)
)
top_ports['label'] = top_ports['dst_ip'] + ':' + top_ports['tcp_dport'].astype(str)

fig = go.Figure(go.Bar(
    x=top_ports['label'], y=top_ports['pkts'],
    marker_color=R,
    text=[f'{v:,}' for v in top_ports['pkts']],
    textposition='outside',
    hovertemplate='%{x}<br>%{y:,} pkts<extra></extra>',
))
fig.update_layout(**LB, title='Top 15 pares IP:porta de destino (TCP)',
    xaxis=dict(title='IP:Porta', tickangle=30, gridcolor=GRID),
    yaxis=dict(title='Pacotes', gridcolor=GRID),
    height=430, showlegend=False)
show(fig)
print(top_ports[['dst_ip', 'tcp_dport', 'pkts']].to_string(index=False))

df_udp = df[df['udp_dport'].notna()].copy()
if len(df_udp):
    print('\nUDP destinos:')
    print(df_udp.groupby(['dst_ip', 'udp_dport']).size()
          .sort_values(ascending=False).head(10).to_string())

     dst_ip  tcp_dport   pkts
 172.18.0.5      30502 416960
172.18.0.11      30501 241530
 172.18.0.4      30503 177525
 172.18.0.8      44035 130329
 172.18.0.2      35929 129298
 172.18.0.6      34137  98745
 172.18.0.9      42769  59377
 172.18.0.9      38145  59335
 172.18.0.6      34865  59335
 172.18.0.2      40385  59110
 172.18.0.2      45697    283
 172.18.0.2      44497    191
 172.18.0.2      39021    191
 172.18.0.2      41901    191
 172.18.0.2      38471    190

UDP destinos:
dst_ip           udp_dport
172.18.0.11      30490.0      108724
224.244.224.245  30490.0       76000
172.18.0.2       30490.0       71925
172.18.0.9       30490.0       14699
172.18.0.7       30490.0       13079
172.18.0.3       30490.0       13013
172.18.0.5       30490.0        2984
172.18.0.4       30490.0        2241
172.18.0.6       30490.0        1496
172.18.0.10      30490.0        1006


## 8. Série Temporal — Intensidade do Tráfego SOME/IP

In [12]:
# Janela de 1 segundo — revela padrão de flood
df_someip['time_bin'] = df_someip['time'].round(0).astype(int)

ts_type = df_someip.groupby(['time_bin', 'msg_type_name']).size().reset_index(name='pkts')
ts_src  = df_someip.groupby(['time_bin', 'src_ip']).size().reset_index(name='pkts')

fig = make_subplots(rows=2, cols=1,
    subplot_titles=['pkt/s por Message Type', 'pkt/s por IP Fonte'],
    shared_xaxes=True, vertical_spacing=0.1)

for mt, color in [('NOTIFICATION', R), ('REQUEST', A), ('RESPONSE', G),
                   ('REQUEST_NO_RETURN', Y), ('ERROR', O)]:
    sub = ts_type[ts_type['msg_type_name'] == mt]
    if len(sub):
        fig.add_trace(go.Scatter(
            x=sub['time_bin'], y=sub['pkts'], name=mt, mode='lines',
            line=dict(color=color, width=1),
            hovertemplate=f'{mt}: %{{y}} pkt/s<extra></extra>'
        ), row=1, col=1)

src_ips = ts_src.groupby('src_ip')['pkts'].sum().nlargest(5).index
colors_src = [R, A, G, Y, P]
for i, ip in enumerate(src_ips):
    sub = ts_src[ts_src['src_ip'] == ip]
    fig.add_trace(go.Scatter(
        x=sub['time_bin'], y=sub['pkts'], name=ip, mode='lines',
        line=dict(color=colors_src[i], width=1),
        hovertemplate=f'{ip}: %{{y}} pkt/s<extra></extra>'
    ), row=2, col=1)

fig.update_layout(**LB, title='Série temporal — SOME/IP pkt/s',
    height=600,
    xaxis2=dict(title='Tempo (s)', gridcolor=GRID),
    yaxis=dict(title='pkt/s', gridcolor=GRID),
    yaxis2=dict(title='pkt/s', gridcolor=GRID),
    legend=dict(bgcolor=PANEL, bordercolor=GRID))
show(fig)

peak = df_someip.groupby('time_bin').size()
print(f'Pico máximo : {peak.max():,} pkt/s  no segundo t={peak.idxmax()}')
print(f'Media       : {peak.mean():.0f} pkt/s')

Pico máximo : 693 pkt/s  no segundo t=745
Media       : 652 pkt/s


## 9. Resumo do Ataque

In [13]:
dur = df['time'].max()
print('=' * 68)
print('RESUMO — dos_noti_flood.pcap — dados derivados do PCAP via tshark')
print('=' * 68)
print(f'  Arquivo         : {PCAP}')
print(f'  Total frames    : {len(df):,}')
print(f'  Total bytes     : {df["frame_len"].sum():,.0f}')
print(f'  Duracao         : {dur:.1f} s')
print(f'  Taxa media      : {len(df)/dur:.0f} pkt/s')
print()
print(f'  SOME/IP frames  : {len(df_someip):,}')
print(f'  SOME/IP-SD      : {len(df_sd):,}')
print()
print('  Message Types observados:')
for mt, cnt in df_someip['msg_type_name'].value_counts().items():
    pct = cnt / len(df_someip) * 100
    print(f'    {mt:<22} {cnt:>10,}  ({pct:.1f}%)')
print()
print('  Service IDs observados:')
for sid, cnt in df_someip['svc_id'].value_counts().items():
    print(f'    {sid}  ->  {cnt:,} pacotes')
print()
print('  Top IPs por volume enviado (SOME/IP):')
top_src = df_someip.groupby('src_ip').size().nlargest(5)
for ip, cnt in top_src.items():
    print(f'    {ip:<18} {cnt:>10,}')
print()
print('  IPs unicos:', df_ip['src_ip'].nunique())
print('  TTL unicos:', sorted(df_ip['ttl'].dropna().unique().astype(int).tolist()))
print()
peak = df_someip.groupby('time_bin').size()
print(f'  Pico de trafego : {peak.max():,} pkt/s')
print(f'  Media trafego   : {peak.mean():.0f} pkt/s')

RESUMO — dos_noti_flood.pcap — dados derivados do PCAP via tshark
  Arquivo         : C:\Mestrado\SDV_Research\experiments\notebooks\data\pcap\dos_noti_flood.pcap
  Total frames    : 1,931,593
  Total bytes     : 163,862,346
  Duracao         : 1493.3 s
  Taxa media      : 1294 pkt/s

  SOME/IP frames  : 973,422
  SOME/IP-SD      : 305,914

  Message Types observados:
    NOTIFICATION              970,828  (99.7%)
    REQUEST                     1,632  (0.2%)
    RESPONSE                      962  (0.1%)

  Service IDs observados:
    0x1002  ->  418,720 pacotes
    0xffff  ->  305,914 pacotes
    0x1003  ->  178,867 pacotes
    0x1001  ->  69,921 pacotes

  Top IPs por volume enviado (SOME/IP):
    172.18.0.5            421,893
    172.18.0.11           251,860
    172.18.0.4            181,311
    172.18.0.2             73,557
    172.18.0.9             14,702



  IPs unicos: 10


  TTL unicos: [1, 64]

  Pico de trafego : 693 pkt/s
  Media trafego   : 652 pkt/s
